# Lecture 8 — Exercises 1-4: Linear Regression with Multiple Variables

Adapted from the Machine Learning Specialization labs by Andrew Ng (DeepLearning.AI and Stanford Online).

This notebook goes with the Lecture 8 slides. It extends linear regression from one feature to several and contains Exercises 1 to 4. Exercise 5 is in the notebook `MI-L08-vectorization`.

Before you start, save your own copy with File > Save a copy in Drive.

Complete the lines marked `# TODO` and run the cells from top to bottom. Most of these lines hold `None`, Python's empty value (like `null` in Java), until you replace it. In Exercises 1, 3 and 4 you also change a value in a cell and run it again for each row of a table.

After each exercise, the cell marked `# Check your result` prints `Check passed`. Most checks stop with an error until you have done the exercise. A check uses `assert condition, message`, which stops with the message when the condition is false. To go on before you finish an exercise, select the cell after the check and use Runtime > Run after.

## 1 - Introduction

### 1.1 Goals
- Extend the linear regression of Lecture 7 from one feature to several
- Scale the features with z-score normalization and see how it changes gradient descent
- Build new features from the existing ones, polynomial features among them
- Measure overfitting with a validation set

### 1.2 Packages
The cell below imports the packages that this notebook uses.
- [NumPy](https://numpy.org) is the fundamental package for working with matrices in Python.
- [matplotlib](https://matplotlib.org) is the library we use to plot graphs in Python.

In [ ]:
import copy, math
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=2)  # print NumPy arrays with two decimals

In [ ]:
def load_house_data():
    """
    Loads houses.txt, with 100 houses. Columns 0 to 3 are the size in sqft, the number of
    bedrooms, the number of floors and the age in years. Column 4 is the price in $1,000.
    The file has no header line, so all 100 houses are read.
    It tries the copy in the course repository on GitHub first and then the file
    data/houses.txt in the folder where the notebook runs. In Colab that folder is /content.

    Returns:
      X (ndarray (m,4)): size, bedrooms, floors and age of each house
      y (ndarray (m,)) : price of each house
    """
    url = "https://raw.githubusercontent.com/andresmasegosa/MI_E26/main/lecture08/data/houses.txt"
    local_file = "data/houses.txt"
    try:
        data = np.loadtxt(url, delimiter=',')
        source = "GitHub"
    except Exception:
        try:
            data = np.loadtxt(local_file, delimiter=',')
            source = local_file
        except Exception:
            raise FileNotFoundError(
                "Could not load houses.txt from GitHub or from data/houses.txt. "
                "Check your internet connection, or put houses.txt in a folder called data "
                "in the folder where the notebook runs. In Colab, upload it to /content/data.") from None
    print(f"Loaded {data.shape[0]} houses from {source}")

    X = data[:,:4]
    y = data[:,4]

    # One house has 0 bedrooms. Set it to 1 so that size/bedrooms is defined.
    X[X[:,1]==0,1]=1

    return X, y

### 1.3 Notation
Here is a summary of some of the notation you will encounter, updated for multiple features.

- General notation
    - $a$ is a scalar, non bold
    - $\mathbf{a}$ is a vector, bold
    - $\mathbf{A}$ is a matrix, bold capital
- Regression
    - $\mathbf{X}$ is the matrix of training examples, `X_train`
    - $\mathbf{y}$ is the vector of training targets, `y_train`
    - $\mathbf{x}^{(i)}$ and $y^{(i)}$ are the $i$-th training example and its target, `X_train[i-1]` and `y_train[i-1]`
    - $m$ is the number of training examples, `m`
    - $n$ is the number of features in each example, `n`
    - $\mathbf{w}$ is the weight vector, `w`
    - $b$ is the bias, `b`
    - $f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = \mathbf{w} \cdot \mathbf{x}^{(i)} + b$ is the prediction of the model for example $i$, `f_wb`

In the formulas we count from 1, as in the slides. Examples go from $i = 1$ to $m$ and features from $j = 1$ to $n$. Python counts from 0, so $\mathbf{x}^{(i)}$ is `X_train[i-1]`.

## 2 - Problem Statement

### 2.1 Dataset
We will use the motivating example of housing price prediction. The dataset contains 100 houses with 4 features (size, bedrooms, floors and age) and the price of each house in thousands of dollars. In this notebook the size is in sqft, while the Lecture 7 notebook used 1,000 sqft.

We would like to build a linear regression model that uses these four features to predict the price of other houses, for example a house of 1,200 sqft with 3 bedrooms and 1 floor, built 40 years ago. Section 6.3 makes that prediction.

One house in the file has 0 bedrooms. `load_house_data` sets it to 1, because Section 7 divides the size by the number of bedrooms.

In [ ]:
# load the dataset
X_train, y_train = load_house_data()
X_features = ['size','bedrooms','floors','age']

The cell below plots the price against each feature.

In [ ]:
fig,ax=plt.subplots(1, 4, figsize=(12, 3), sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X_train[:,i],y_train)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("Price in $1,000")
plt.show()

### 2.2 Matrix X containing our examples
The examples are stored in a NumPy matrix `X_train`. Each row of the matrix is one example. With $m$ training examples ($m = 100$ here) and $n$ features ($n = 4$ here), $\mathbf{X}$ is a matrix with dimensions ($m$, $n$), that is $m$ rows and $n$ columns.


$$\mathbf{X} =
\begin{pmatrix}
 x^{(1)}_1 & x^{(1)}_2 & \cdots & x^{(1)}_{n} \\
 x^{(2)}_1 & x^{(2)}_2 & \cdots & x^{(2)}_{n} \\
 \cdots \\
 x^{(m)}_1 & x^{(m)}_2 & \cdots & x^{(m)}_{n}
\end{pmatrix}
$$

- $\mathbf{x}^{(i)} = (x^{(i)}_1, x^{(i)}_2, \cdots, x^{(i)}_n)$ is the vector with example $i$.
- $x^{(i)}_j$ is element $j$ of example $i$. The superscript in parentheses is the example number and the subscript is the feature number.

For example, $x^{(1)}_1$, the size of the first house, is `X_train[0, 0]`, and $x^{(i)}_j$ is `X_train[i-1, j-1]`.

The `print` lines use f-strings. In `f"Cost: {cost:0.2f}"` Python writes the value of `cost` in place of the braces, with two decimals.

The cell below prints the shape and the type of `X_train` and `y_train`.

In [ ]:
# m houses (rows) and n features (columns)
m, n = X_train.shape
print(f"X Shape: {X_train.shape}, X Type: {type(X_train)}")
print(f"y Shape: {y_train.shape}, y Type: {type(y_train)}")

### 2.3 Parameter vector w, b

* $\mathbf{w}$ is a vector with $n$ elements, one weight per feature. Here $n = 4$. We write it as a column vector.

$$\mathbf{w} = \begin{pmatrix}
w_1 \\
w_2 \\
\cdots\\
w_n
\end{pmatrix}
$$
* $b$ is a scalar parameter.

The values of `w_init` and `b_init` below are arbitrary. We only use them to test the functions in Sections 3 to 5.

In [ ]:
# arbitrary parameters, used to test the functions below
b_init = 785.1811367994083
w_init = np.array([ 0.39133535, 18.75376741, -53.36032453, -26.42131618])
print(f"w_init shape: {w_init.shape}, b_init type: {type(b_init)}")

## 3 - Model Prediction With Multiple Variables
The model's prediction with multiple variables is given by the linear model:

$$ f_{\mathbf{w},b}(\mathbf{x}) =  w_1x_1 + w_2x_2 + \cdots + w_nx_n + b \tag{1}$$
or in vector notation:
$$ f_{\mathbf{w},b}(\mathbf{x}) = \mathbf{w} \cdot \mathbf{x} + b  \tag{2} $$
where $\cdot$ is the dot product of two vectors.

The function `compute_model_output` below computes equation (2) for every house, one house at a time, with `np.dot`. It is the function of Exercise 1 of Lecture 7 with a vector $\mathbf{w}$.

In [ ]:
def compute_model_output(X, w, b):
    """
    Computes the prediction of a linear model for every example, one example at a time
    Args:
      X (ndarray (m,n)): Data, m examples with n features
      w (ndarray (n,)) : model parameters
      b (scalar)       : model parameter

    Returns:
      f_wb (ndarray (m,)): model predictions
    """
    m = X.shape[0]
    f_wb = np.zeros(m)
    for i in range(m):
        f_wb[i] = np.dot(X[i], w) + b
    return f_wb

In [ ]:
# get a row from our training data
x_vec = X_train[0,:]
print(f"x_vec shape {x_vec.shape}, x_vec value: {x_vec}")

# make a prediction for every house, then look at the first one
f_wb = compute_model_output(X_train, w_init, b_init)
print(f"f_wb shape {f_wb.shape}, first prediction: {f_wb[0]:0.2f}, target value: {y_train[0]}")

Expected output:

    x_vec shape (4,), x_vec value: [952.   2.   1.  65.]
    f_wb shape (100,), first prediction: -575.51, target value: 271.5

The prediction is negative because `w_init` and `b_init` are arbitrary values, not fitted ones.

## 4 - Compute Cost With Multiple Variables
The equation for the cost function with multiple variables $J(\mathbf{w},b)$ is:
$$J(\mathbf{w},b) = \frac{1}{2m} \sum\limits_{i = 1}^{m} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})^2 \tag{3}$$
where:
$$ f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = \mathbf{w} \cdot \mathbf{x}^{(i)} + b  \tag{4} $$


In the Lecture 7 notebook $w$ and $x^{(i)}$ were scalars. Here $\mathbf{w}$ and $\mathbf{x}^{(i)}$ are vectors with one entry per feature.

The cost $J$ is half the mean squared error. So $\sqrt{2J}$ is the size of a typical prediction error, in thousands of dollars.

In [ ]:
def compute_cost(X, y, w, b):
    """
    compute cost
    Args:
      X (ndarray (m,n)): Data, m examples with n features
      y (ndarray (m,)) : target values
      w (ndarray (n,)) : model parameters
      b (scalar)       : model parameter

    Returns:
      cost (scalar): cost
    """
    m = X.shape[0]
    cost = 0.0
    for i in range(m):
        f_wb_i = np.dot(X[i], w) + b           # the dot product of two (n,) vectors is a number
        cost = cost + (f_wb_i - y[i])**2
    cost = cost / (2 * m)
    return cost

In [ ]:
# Compute and display the cost at the arbitrary parameters w_init, b_init
cost = compute_cost(X_train, y_train, w_init, b_init)
print(f'Cost at w_init, b_init: {cost:0.2f}')

Expected output:

    Cost at w_init, b_init: 223658.99

## 5 - Gradient Descent With Multiple Variables
As in Lecture 7, gradient descent repeats these updates until convergence.

$$\begin{align*} \text{repeat}&\text{ until convergence:} \; \lbrace \newline\;
& w_j := w_j -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial w_j} \tag{5}  \; & \text{for } j = 1, \dots, n\newline
&b\ \ := b -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial b}  \newline \rbrace
\end{align*}$$

where $n$ is the number of features, the parameters $w_j$ and $b$ are updated simultaneously, and

$$
\begin{align}
\frac{\partial J(\mathbf{w},b)}{\partial w_j}  &= \frac{1}{m} \sum\limits_{i = 1}^{m} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})x_{j}^{(i)} \tag{6}  \\
\frac{\partial J(\mathbf{w},b)}{\partial b}  &= \frac{1}{m} \sum\limits_{i = 1}^{m} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}) \tag{7}
\end{align}
$$
* $m$ is the number of training examples in the dataset.


* $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ is the model's prediction, and $y^{(i)}$ is the target value.

`compute_gradient` below returns `dj_db` first and then `dj_dw`. The Lecture 7 notebook returned them in the other order.

In [ ]:
def compute_gradient(X, y, w, b):
    """
    Computes the gradient for linear regression
    Args:
      X (ndarray (m,n)): Data, m examples with n features
      y (ndarray (m,)) : target values
      w (ndarray (n,)) : model parameters
      b (scalar)       : model parameter

    Returns:
      dj_db (scalar):       The partial derivative of the cost with respect to b
      dj_dw (ndarray (n,)): The partial derivatives of the cost with respect to the weights in w
    """
    m,n = X.shape           #(number of examples, number of features)
    dj_dw = np.zeros((n,))
    dj_db = 0.

    for i in range(m):
        err = (np.dot(X[i], w) + b) - y[i]
        for j in range(n):
            dj_dw[j] = dj_dw[j] + err * X[i, j]
        dj_db = dj_db + err
    dj_dw = dj_dw / m
    dj_db = dj_db / m

    return dj_db, dj_dw

In [ ]:
# Compute and display the gradient at w_init, b_init
tmp_dj_db, tmp_dj_dw = compute_gradient(X_train, y_train, w_init, b_init)
print(f'dj_db at w_init, b_init: {tmp_dj_db:0.2f}')
print(f'dj_dw at w_init, b_init: {tmp_dj_dw}')

Expected output:

    dj_db at w_init, b_init: -67.63
    dj_dw at w_init, b_init: [ 6.45e+03 -1.15e+02 -6.80e+00 -1.96e+04]

In [ ]:
def gradient_descent(X, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters):
    """
    Performs batch gradient descent to learn w and b. Updates w and b by taking
    num_iters gradient steps with learning rate alpha

    Args:
      X (ndarray (m,n))   : Data, m examples with n features
      y (ndarray (m,))    : target values
      w_in (ndarray (n,)) : initial model parameters
      b_in (scalar)       : initial model parameter
      cost_function       : function to compute cost
      gradient_function   : function to compute the gradient
      alpha (float)       : Learning rate
      num_iters (int)     : number of iterations to run gradient descent

    Returns:
      w (ndarray (n,))    : Updated values of parameters
      b (scalar)          : Updated value of parameter
      J_history (list)    : cost after each iteration
      """

    # A list to store the cost J at each iteration, for the plots later
    J_history = []
    w = copy.deepcopy(w_in)
    b = b_in

    for i in range(num_iters):

        # Calculate the gradient at the current w and b
        dj_db, dj_dw = gradient_function(X, y, w, b)

        # Update the parameters using alpha and the gradient
        w = w - alpha * dj_dw
        b = b - alpha * dj_db

        # Save cost J at each iteration
        if i<100000:      # prevent resource exhaustion
            J_history.append( cost_function(X, y, w, b))

        # Print the cost 10 times during the run, or at every iteration if num_iters < 10
        if i% math.ceil(num_iters / 10) == 0:
            print(f"Iteration {i:4d}: Cost {J_history[-1]:8.2f}   ")

    return w, b, J_history

## 6 - Feature Scaling

### 6.1 Running Gradient Descent without Feature Scaling

We first run gradient descent on the raw features. The learning rate is very small, $\alpha = 10^{-7}$, and we run 10,000 iterations. As in Lecture 7, the line for iteration 0 shows the cost after the first step.

A larger learning rate does not work here. With $\alpha = 0.1$, the value used in Section 6.3, gradient descent diverges on the raw features and the cost grows until it becomes `inf` and then `nan`. The reason is the size of the house. It is in the thousands, so the partial derivative for its weight is very large, and a step of $\alpha$ times that derivative jumps far past the minimum.

In [ ]:
# initialize parameters
initial_w = np.zeros(n)
initial_b = 0.
# some gradient descent settings
iterations = 10000
alpha = 1e-7    # 1e-7 is 0.0000001
# run gradient descent
w_final, b_final, J_hist = gradient_descent(X_train, y_train, initial_w, initial_b,
                                                    compute_cost, compute_gradient,
                                                    alpha, iterations)
print(f"b,w found by gradient descent: {b_final:0.2f},{w_final} ")
print(f"Cost after {iterations} iterations: {J_hist[-1]:0.2f}")
for i in range(3):
    print(f"prediction: {np.dot(X_train[i], w_final) + b_final:0.2f}, target value: {y_train[i]}")

Expected output:

    Iteration    0: Cost 44154.43   
    Iteration 1000: Cost  1530.43   
    Iteration 2000: Cost  1498.06   
    Iteration 3000: Cost  1471.15   
    Iteration 4000: Cost  1448.76   
    Iteration 5000: Cost  1430.15   
    Iteration 6000: Cost  1414.67   
    Iteration 7000: Cost  1401.79   
    Iteration 8000: Cost  1391.08   
    Iteration 9000: Cost  1382.18   
    b,w found by gradient descent: 0.01,[ 0.26 -0.   -0.01 -0.42] 
    Cost after 10000 iterations: 1374.78
    prediction: 223.19, target value: 271.5
    prediction: 300.45, target value: 300.0
    prediction: 505.18, target value: 509.8

The cost is still going down slowly, and it is far above 231.05, the cost that Section 6.3 reaches.

In [ ]:
# plot cost versus iteration
plt.plot(J_hist)
plt.title("Cost vs. iteration")
plt.ylabel('Cost')
plt.xlabel('iteration step')
plt.show()

In [ ]:
# predict the targets with the parameters found on the raw features
yp = compute_model_output(X_train, w_final, b_final)

# plot predictions and targets versus original features
fig,ax=plt.subplots(1,4,figsize=(12, 3),sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X_train[:,i],y_train, label = 'target')
    ax[i].set_xlabel(X_features[i])
    ax[i].scatter(X_train[:,i],yp,color="orange", label = 'predict')
ax[0].set_ylabel("Price")
ax[0].legend()
fig.suptitle("target versus prediction")
plt.show()

### 6.2 Z-score normalization

Gradient descent uses one learning rate for all the weights. The size is in the thousands and the number of bedrooms is between 1 and 4. A learning rate small enough for the weight of the size is far too small for the other weights, so gradient descent needs a very long time. Feature scaling puts all the features on a similar scale, and then one learning rate suits all of them.

After z-score normalization, all features have a mean of 0 and a standard deviation of 1.

Z-score normalization replaces each value of feature $j$ by
$$x^{(i)}_j := \dfrac{x^{(i)}_j - \mu_j}{\sigma_j} \tag{8}$$
where $j$ selects a feature or a column in the $\mathbf{X}$ matrix. $\mu_j$ is the mean of all the values for feature $j$ and $\sigma_j$ is the standard deviation of feature $j$.
$$
\begin{align}
\mu_j &= \frac{1}{m} \sum_{i=1}^{m} x^{(i)}_j \tag{9}\\
\sigma^2_j &= \frac{1}{m} \sum_{i=1}^{m} (x^{(i)}_j - \mu_j)^2  \tag{10}
\end{align}
$$

> When you scale the features, store the mean and the standard deviation that you used. After training we often want to predict the price of a house we have not seen before. Given a new $\mathbf{x}$ (size, bedrooms, floors and age), we must first scale it with the mean and the standard deviation computed from the training set. Sections 6.3 and 8.2 follow this rule.

In [ ]:
def zscore_normalize_features(X):
    """
    Scales each column of X with z-score normalization

    Args:
      X (ndarray (m,n))     : input data, m examples, n features

    Returns:
      X_norm (ndarray (m,n)): X with every column scaled
      X_mean (ndarray (n,)) : mean of each feature
      X_sigma (ndarray (n,)): standard deviation of each feature
    """
    # find the mean of each column/feature
    X_mean     = np.mean(X, axis=0)                 # X_mean has shape (n,)
    # find the standard deviation of each column/feature
    X_sigma  = np.std(X, axis=0)                  # X_sigma has shape (n,)
    # NumPy subtracts X_mean from every row of X and divides every row by X_sigma. This is called broadcasting.
    X_norm = (X - X_mean) / X_sigma

    return (X_norm, X_mean, X_sigma)

The cell below scales the four features. It keeps their means in `X_mean` and their standard deviations in `X_sigma`, and prints them.

In [ ]:
X_norm, X_mean, X_sigma = zscore_normalize_features(X_train)
for name, mu_j, sigma_j in zip(X_features, X_mean, X_sigma):
    print(f"{name:9s} mean {mu_j:8.2f}   standard deviation {sigma_j:7.2f}")

The plots below show the price against each scaled feature. Compare them with the plots of Section 2.1.

In [ ]:
fig,ax=plt.subplots(1, 4, figsize=(12, 3), sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X_norm[:,i],y_train)
    ax[i].set_xlabel(X_features[i])

ax[0].set_ylabel("Price in $1,000")
fig.suptitle("Data after feature scaling")
plt.show()

The plots below show the size and the age of every house in three steps. The first plot uses the raw values. The second subtracts the mean, and the third is the full z-score. Look at the scale of the axes.

In [ ]:
X_centered = X_train - X_mean

fig,ax=plt.subplots(1, 3, figsize=(12, 3))
ax[0].scatter(X_train[:,0], X_train[:,3])
ax[0].set_xlabel(X_features[0])
ax[0].set_ylabel(X_features[3])
ax[0].set_title("raw")
ax[0].axis('equal')

ax[1].scatter(X_centered[:,0], X_centered[:,3])
ax[1].set_xlabel(X_features[0])
ax[1].set_ylabel(X_features[3])
ax[1].set_title(r"X - $\mu$")
ax[1].axis('equal')

ax[2].scatter(X_norm[:,0], X_norm[:,3])
ax[2].set_xlabel(X_features[0])
ax[2].set_ylabel(X_features[3])
ax[2].set_title("Z-score normalized")
ax[2].axis('equal')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
fig.suptitle("Size and age before and after z-score normalization")
plt.show()

### 6.3 Running Gradient Descent after Feature Scaling

In [ ]:
# initialize parameters
initial_w = np.zeros(n)
initial_b = 0.
# some gradient descent settings
iterations = 1000
alpha = 0.1
# run gradient descent
w_final, b_final, J_hist = gradient_descent(X_norm, y_train, initial_w, initial_b,
                                                    compute_cost, compute_gradient,
                                                    alpha, iterations)
print(f"b,w found by gradient descent: {b_final:0.2f},{w_final} ")
print(f"Cost after {iterations} iterations: {J_hist[-1]:0.2f}")
for i in range(3):
    print(f"prediction: {np.dot(X_norm[i], w_final) + b_final:0.2f}, target value: {y_train[i]}")

Expected output:

    Iteration    0: Cost 57299.69   
    Iteration  100: Cost   234.01   
    Iteration  200: Cost   231.06   
    Iteration  300: Cost   231.05   
    Iteration  400: Cost   231.05   
    Iteration  500: Cost   231.05   
    Iteration  600: Cost   231.05   
    Iteration  700: Cost   231.05   
    Iteration  800: Cost   231.05   
    Iteration  900: Cost   231.05   
    b,w found by gradient descent: 362.24,[111.63 -21.35 -33.16 -37.92] 
    Cost after 1000 iterations: 231.05
    prediction: 249.29, target value: 271.5
    prediction: 295.28, target value: 300.0
    prediction: 486.48, target value: 509.8

The cost stops changing after about 300 iterations. It is the lowest cost that this model can reach with these four features.

In [ ]:
# plot cost versus iteration
plt.plot(J_hist)
plt.title("Cost vs. iteration")
plt.ylabel('Cost')
plt.xlabel('iteration step')
plt.show()

In [ ]:
# predict the targets with the parameters found on the scaled features
yp = compute_model_output(X_norm, w_final, b_final)

# plot predictions and targets versus original features
fig,ax=plt.subplots(1,4,figsize=(12, 3),sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X_train[:,i],y_train, label = 'target')
    ax[i].set_xlabel(X_features[i])
    ax[i].scatter(X_train[:,i],yp,color="orange", label = 'predict')
ax[0].set_ylabel("Price")
ax[0].legend()
fig.suptitle("target versus prediction")
plt.show()

The cell below predicts the price of the house of Section 2.1, with 1,200 sqft, 3 bedrooms and 1 floor, built 40 years ago. The model was trained on scaled features, so the new house is first scaled with the mean and the standard deviation of the training set.

In [ ]:
# the house of Section 2.1, with 1,200 sqft, 3 bedrooms and 1 floor, built 40 years ago
x_house = np.array([1200, 3, 1, 40])
# scale it with the mean and the standard deviation of the training set
x_house_norm = (x_house - X_mean) / X_sigma
print(f"Predicted price of the new house: {np.dot(x_house_norm, w_final) + b_final:0.2f} thousand dollars")

Expected output:

    Predicted price of the new house: 318.66 thousand dollars

### Exercise 1: Feature Scaling

In this exercise you compare how the learning rate behaves with and without feature scaling.

- Run the cell below once for every row of the table in the answer cell. Set `use_scaled_features` and `alpha` to the values of that row.
- On the raw features try `alpha` equal to `1e-7`, `5e-7` and `1e-6`. In Python `1e-7` means $1 \times 10^{-7}$, and `5e-7` means $5 \times 10^{-7}$.
- On the scaled features try `alpha` equal to `0.001`, `0.01`, `0.1`, `0.5` and `1.0`.
- Every run uses 1,000 iterations, so the costs can be compared. The lowest cost this model can reach is 231.05, the cost that Section 6.3 converges to.
- For each run, write down the cost after 1,000 iterations and whether gradient descent has converged. If it has not, say whether it is still going down slowly or has diverged.
- A run that diverges prints very large costs. The last line then writes the cost as a power of ten, for example `3.20e+50` for $3.2 \times 10^{50}$.
- At the end, write the largest $\alpha$ of each list that does not diverge in the cell after the answer cell, and run the check.

Then answer these questions.

1. What is the largest $\alpha$ in each list that does not diverge?
2. Section 6.2 says why one learning rate cannot suit the raw features. Which rows of your table show it? Which plots of Sections 2.1 and 6.2 show why?
3. Explain why feature scaling does not change the lowest cost that the model can reach.

In [ ]:
# Exercise 1: set these two values for each row of the table, then run the cell
use_scaled_features = False   # False: raw features X_train. True: scaled features X_norm
alpha = 1e-7

# X_ex1 is X_norm if use_scaled_features is True, and X_train otherwise
X_ex1 = X_norm if use_scaled_features else X_train
w_ex1, b_ex1, J_ex1 = gradient_descent(X_ex1, y_train, np.zeros(n), 0.,
                                       compute_cost, compute_gradient, alpha, 1000)
if J_ex1[-1] < 1e6:
    print(f"Cost after 1000 iterations: {J_ex1[-1]:0.2f}")
else:
    print(f"Cost after 1000 iterations: {J_ex1[-1]:0.2e}")

plt.plot(J_ex1)
plt.yscale("log")
plt.title(f"Cost vs. iteration, alpha = {alpha}, scaled features: {use_scaled_features}")
plt.ylabel('Cost (log scale)')
plt.xlabel('iteration step')
plt.show()

Expected output:

    Iteration    0: Cost 44154.43   
    Iteration  100: Cost  1565.13   
    Iteration  200: Cost  1560.99   
    Iteration  300: Cost  1556.92   
    Iteration  400: Cost  1552.92   
    Iteration  500: Cost  1549.00   
    Iteration  600: Cost  1545.15   
    Iteration  700: Cost  1541.36   
    Iteration  800: Cost  1537.65   
    Iteration  900: Cost  1534.01   
    Cost after 1000 iterations: 1530.47

This is the output for the values already in the cell, the raw features and `alpha = 1e-7`.

Write your results and answers here.

| Features | $\alpha$ | Cost after 1000 iterations | Converged, slow or diverged? |
|---|---|---|---|
| raw | 1e-7 | | |
| raw | 5e-7 | | |
| raw | 1e-6 | | |
| scaled | 0.001 | | |
| scaled | 0.01 | | |
| scaled | 0.1 | | |
| scaled | 0.5 | | |
| scaled | 1.0 | | |

1. ...
2. ...
3. ...

In [ ]:
# Exercise 1: write down the largest learning rates that did not diverge
alpha_max_raw = None     # TODO: the largest alpha in the raw-feature list that does not diverge
alpha_max_scaled = None  # TODO: the largest alpha in the scaled-feature list that does not diverge

In [ ]:
# Check your result
import contextlib, io
assert alpha_max_raw is not None and alpha_max_scaled is not None, "Write the two learning rates in the cell above and run it."

def cost_goes_down(X, alpha):
    # 1,000 iterations from w = 0, b = 0. True if the cost stays finite and goes down.
    # The with block runs gradient descent without printing and without overflow warnings.
    with contextlib.redirect_stdout(io.StringIO()), np.errstate(all="ignore"):
        J = gradient_descent(X, y_train, np.zeros(n), 0., compute_cost, compute_gradient, alpha, 1000)[2]
    return bool(np.all(np.isfinite(J)) and J[-1] < J[0])

for name, alpha_max, X_check, alphas in [
        ("alpha_max_raw", alpha_max_raw, X_train, [1e-7, 5e-7, 1e-6]),
        ("alpha_max_scaled", alpha_max_scaled, X_norm, [0.001, 0.01, 0.1, 0.5, 1.0])]:
    # the position of alpha_max in the list of the exercise
    k = [i for i, a in enumerate(alphas) if np.isclose(alpha_max, a, rtol=1e-6, atol=0)]
    assert k, f"{name} should be one of the values in its list in the exercise."
    k = k[0]
    assert cost_goes_down(X_check, alphas[k]), f"With {name} gradient descent diverges. Choose a smaller value from the list."
    if k + 1 < len(alphas):
        assert not cost_goes_down(X_check, alphas[k + 1]), f"{name} is not the largest value that works. The next value in the list does not diverge either."
print(f"With scaling the learning rate can be {alpha_max_scaled / alpha_max_raw:,.0f} times larger.")
print("Check passed")

## 7 - Feature Engineering

We add a new feature to the training data. It is the size of the house divided by its number of bedrooms.

$$ \text{size\_per\_bedroom} = \frac{\text{size}}{\text{number of bedrooms}}$$

In [ ]:
# Create a larger matrix
X_train_extended = np.zeros((X_train.shape[0],X_train.shape[1]+1))

# Copy the training data to the new matrix
X_train_extended[:,:-1] = X_train

# Add the new column
X_train_extended[:,-1] = X_train[:,0]/X_train[:,1]

# The names of the five features
X_features_extended = ['size','bedrooms','floors','age','size_per_bedroom']

The cell below plots the price against each of the five features.

In [ ]:
# plot the targets versus every feature, the new one included
fig,ax=plt.subplots(1,len(X_features_extended),figsize=(12, 3),sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X_train_extended[:,i],y_train, label = 'target')
    ax[i].set_xlabel(X_features_extended[i])
ax[0].set_ylabel("Price")
ax[0].legend()
fig.suptitle("Extended Data")
plt.show()

The cell below scales the five features.

In [ ]:
X_ext_norm, X_ext_mean, X_ext_sigma = zscore_normalize_features(X_train_extended)

The cell below runs gradient descent on the five scaled features, with $\alpha = 0.1$ and 1,000 iterations as in Section 6.3.

In [ ]:
# initialize parameters
initial_w = np.zeros(X_ext_norm.shape[1])
initial_b = 0.
# some gradient descent settings
iterations = 1000
alpha = 0.1
# run gradient descent
w_final, b_final, J_hist = gradient_descent(X_ext_norm, y_train, initial_w, initial_b,
                                                    compute_cost, compute_gradient,
                                                    alpha, iterations)
print(f"b,w found by gradient descent: {b_final:0.2f},{w_final} ")
print(f"Cost after {iterations} iterations: {J_hist[-1]:0.2f}")
for i in range(3):
    print(f"prediction: {np.dot(X_ext_norm[i], w_final) + b_final:0.2f}, target value: {y_train[i]}")

Expected output:

    Iteration    0: Cost 56965.79   
    Iteration  100: Cost   278.25   
    Iteration  200: Cost   246.65   
    Iteration  300: Cost   234.74   
    Iteration  400: Cost   230.26   
    Iteration  500: Cost   228.56   
    Iteration  600: Cost   227.93   
    Iteration  700: Cost   227.68   
    Iteration  800: Cost   227.59   
    Iteration  900: Cost   227.56   
    b,w found by gradient descent: 362.24,[104.43 -14.65 -32.67 -37.47   6.79] 
    Cost after 1000 iterations: 227.55
    prediction: 247.17, target value: 271.5
    prediction: 296.27, target value: 300.0
    prediction: 485.40, target value: 509.8

With the four original features the cost was 231.05.

The cell below plots the predictions and the prices against each feature.

In [ ]:
# predict the targets with the five scaled features
yp = compute_model_output(X_ext_norm, w_final, b_final)

# plot predictions and targets versus original features
fig,ax=plt.subplots(1,X_train_extended.shape[1],figsize=(12, 3),sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X_train_extended[:,i],y_train, label = 'target')
    ax[i].set_xlabel(X_features_extended[i])
    ax[i].scatter(X_train_extended[:,i],yp,color="orange", label = 'predict')
ax[0].set_ylabel("Price")
ax[0].legend()
fig.suptitle("target versus prediction")
plt.show()

Gradient descent stops after a fixed number of iterations, so its cost can stay above the lowest cost that the model can reach. The next cell defines the helper function `min_cost`, which computes that lowest cost. It uses `np.linalg.lstsq`, which finds the best parameters directly, without gradient descent. We use it as a reference for gradient descent in Exercises 2 and 3.

In [ ]:
def min_cost(X, y):
    """
    Lowest cost J(w,b) that any w, b can reach on X, y. np.linalg.lstsq finds
    the best w and b directly, without gradient descent.

    Args:
      X (ndarray (m,n)): Data, m examples with n features
      y (ndarray (m,)) : target values

    Returns:
      cost (scalar): the minimum of J(w,b)
    """
    A = np.column_stack([X, np.ones(X.shape[0])])   # a column of ones for b
    params = np.linalg.lstsq(A, y, rcond=None)[0]
    return compute_cost(X, y, params[:-1], params[-1])


print(f"Minimum cost with the five features: {min_cost(X_ext_norm, y_train):0.2f}")

Expected output:

    Minimum cost with the five features: 227.54

Gradient descent reached 227.55 after 1,000 iterations, so it got very close to this minimum.

### Exercise 2: Feature Engineering

- Think of a new feature that could help to predict the price of a house. Build it from the columns of `X_train`, as we did for `size_per_bedroom`. Do not use `y_train`. For a new house we do not know the price. Write the feature and a short name for it in the cell below, and run the check.
- Explain why you chose this feature.
- Run the cell after the check. It adds your feature to the five features of Section 7 and scales all six. Then it runs gradient descent for 1,000 iterations and prints its cost next to the minimum cost. After 1,000 iterations gradient descent can still be above the minimum, so compare the minimum costs. With the five features the minimum cost was 227.54. Does it go down with your feature?
- Run the cell after it. It adds a column of random numbers instead of your feature. Does the minimum cost also go down?
- What does this tell you about using the training cost to choose features?

In [ ]:
# Exercise 2: your new feature, one value per house
new_feature = None               # TODO: an array of shape (m,), computed from the columns of X_train
new_feature_name = "my_feature"  # TODO: a short name for your feature

In [ ]:
# Check your result
assert new_feature is not None, "Write your feature in new_feature in the cell above and run it."
assert np.shape(new_feature) == (m,), f"new_feature should have one value per house, shape ({m},), not {np.shape(new_feature)}"
assert np.all(np.isfinite(new_feature)), "new_feature contains inf or nan. Look for a division by zero."
assert np.std(new_feature) > 0, "new_feature has the same value for every house, so it cannot help the model."
print("Check passed")

In [ ]:
# Add your feature as a sixth column, scale all the columns and run gradient descent
assert new_feature is not None, "Write your feature in new_feature first, two cells above."
# np.column_stack puts the arrays side by side as the columns of one matrix
X_ex2 = np.column_stack([X_train_extended, new_feature])
# _ takes the mean and the standard deviation, which are not needed here
X_ex2_norm, _, _ = zscore_normalize_features(X_ex2)
w_ex2, b_ex2, J_ex2 = gradient_descent(X_ex2_norm, y_train, np.zeros(X_ex2.shape[1]), 0.,
                                       compute_cost, compute_gradient, 0.1, 1000)
print(f"With {new_feature_name}: cost after 1000 iterations {J_ex2[-1]:0.2f}, minimum cost {min_cost(X_ex2_norm, y_train):0.2f}")

In [ ]:
# The same, with a column of random numbers instead of your feature
rng = np.random.default_rng(0)
# rng.normal draws numbers from a normal distribution with mean 0 and standard deviation 1
noise = rng.normal(size=m)
X_noise = np.column_stack([X_train_extended, noise])
X_noise_norm, _, _ = zscore_normalize_features(X_noise)
w_noise, b_noise, J_noise = gradient_descent(X_noise_norm, y_train, np.zeros(X_noise.shape[1]), 0.,
                                             compute_cost, compute_gradient, 0.1, 1000)
print(f"With a column of random numbers: cost after 1000 iterations {J_noise[-1]:0.2f}, minimum cost {min_cost(X_noise_norm, y_train):0.2f}")

Expected output:

    Iteration    0: Cost 56949.58   
    Iteration  100: Cost   272.14   
    Iteration  200: Cost   240.92   
    Iteration  300: Cost   229.14   
    Iteration  400: Cost   224.70   
    Iteration  500: Cost   223.02   
    Iteration  600: Cost   222.39   
    Iteration  700: Cost   222.15   
    Iteration  800: Cost   222.06   
    Iteration  900: Cost   222.02   
    With a column of random numbers: cost after 1000 iterations 222.01, minimum cost 222.00

Write your feature, the reason you chose it and your answers here.

...

## 8 - Overfitting

### 8.1 Adding Polynomial Features

We add polynomial features, such as $\text{size}^2$ and $\text{size}^3$, to the training data and look at what happens to the cost function after running gradient descent.

We scale the four features before we raise the size to a power. The raw size goes up to 3,194 sqft, so $\text{size}^{20}$ would be around $10^{70}$. The scaled size stays between about -1.5 and 4.3, and its powers are much smaller. After adding the powers we scale all the columns again, so that every feature has mean 0 and standard deviation 1 before gradient descent.

The next cell defines the helper function `poly_features`, which adds the columns $\text{size}^2, \dots, \text{size}^{d}$. Exercise 3 compares gradient descent with `min_cost` from Section 7.

In [ ]:
def poly_features(X, degree):
    """
    Adds the columns size^2, ..., size^degree to X. The size is column 0 of X.

    Args:
      X (ndarray (m,n)): Data, m examples with n features
      degree (int)     : highest power of the size

    Returns:
      X_poly (ndarray (m,n+degree-1)): X followed by the new columns
    """
    new_columns = [X[:, 0]**d for d in range(2, degree + 1)]
    return np.column_stack([X] + new_columns)

### Exercise 3: Polynomial Features

- Set `degree` in the first cell below and run the cells down to the check. Do this for degree = 1, 2, 3, 5, 10 and 20.
- For each degree, write down the cost after 1,000 iterations of gradient descent and the minimum cost.
- Keep $\alpha = 0.1$ and do not go above degree 20. At degree 22, $\alpha = 0.1$ is too large and gradient descent diverges.

Then answer these questions.

1. Does the minimum cost go down as the degree grows? Can it ever go up?
2. Does gradient descent reach the minimum cost at every degree? If not, where does it fall behind, and why?
3. Do you think we are doing better by having more polynomial features?

In [ ]:
# Enter the degree of the new polynomial features.
degree = 2

# scale the four features, add size^2 ... size^degree, then scale all the columns
X_scaled, _, _ = zscore_normalize_features(X_train)
X_poly, _, _ = zscore_normalize_features(poly_features(X_scaled, degree))
print(f"X_poly has {X_poly.shape[1]} features")

In [ ]:
# initialize parameters
initial_w = np.zeros(X_poly.shape[1])
initial_b = 0.
# some gradient descent settings
iterations = 1000
alpha = 0.1
# run gradient descent
w_final, b_final, J_hist = gradient_descent(X_poly, y_train, initial_w, initial_b,
                                                    compute_cost, compute_gradient,
                                                    alpha, iterations)
J_min = min_cost(X_poly, y_train)
print(f"Degree {degree}: cost after {iterations} iterations {J_hist[-1]:0.2f}, minimum cost {J_min:0.2f}")

Expected output:

    Iteration    0: Cost 57154.14   
    Iteration  100: Cost   232.23   
    Iteration  200: Cost   221.53   
    Iteration  300: Cost   221.37   
    Iteration  400: Cost   221.37   
    Iteration  500: Cost   221.37   
    Iteration  600: Cost   221.37   
    Iteration  700: Cost   221.37   
    Iteration  800: Cost   221.37   
    Iteration  900: Cost   221.37   
    Degree 2: cost after 1000 iterations 221.37, minimum cost 221.37

This is the output for degree = 2, the value already in the cell.

In [ ]:
# Check your result
assert degree >= 1, "degree should be 1 or more."
assert X_poly.shape == (m, 3 + degree), f"X_poly should have 4 + {degree - 1} columns for degree {degree}. Run the cell that sets the degree."
assert len(w_final) == X_poly.shape[1], "Run the gradient descent cell again after you change the degree."
assert np.isfinite(J_hist[-1]) and J_hist[-1] < J_hist[0], "Gradient descent diverged. Use alpha = 0.1 and a degree of 20 or less."
assert J_hist[-1] >= J_min - 1e-6, "Gradient descent cannot go below the minimum cost"
print("Check passed")

Write your results and answers here.

| Degree | Cost after 1000 iterations | Minimum cost |
|---|---|---|
| 1 | | |
| 2 | | |
| 3 | | |
| 5 | | |
| 10 | | |
| 20 | | |

1. ...
2. ...
3. ...

### 8.2 Measuring Overfitting

To see whether a model overfits, we need houses that the model was not trained on. We split the 100 houses at random into a training set of 66 houses and a validation set of 34 houses. Gradient descent only sees the training set. The validation set tells us how well the model predicts houses it has not seen, and we use it to choose the degree.

The order of the steps matters. We split first. Then we compute the mean and the standard deviation for the z-score on the training rows only, and we apply the same values to the validation rows. This is the rule from Section 6.2. If the validation rows were used to compute the mean and the standard deviation, information about them would leak into training.

The split uses a random number generator with a fixed seed, so that everybody gets the same split and the results can be reproduced. `np.random.default_rng(seed)` makes the generator, and its `permutation(m)` returns the numbers 0 to $m-1$ in a random order.

Because we use the validation set to choose the degree, the validation cost of the chosen model is an optimistic estimate of its error on new houses. A fair final estimate needs a third set, the test set, which is used only once at the end. In the Kaggle competition the hidden half of the test days plays this role.

In [ ]:
# Split the houses at random into 66 for training and 34 for validation
seed = 8
idx = np.random.default_rng(seed).permutation(m)
size_tr = 66
idx_tr, idx_val = idx[:size_tr], idx[size_tr:]

X_tr_raw, y_tr = X_train[idx_tr], y_train[idx_tr]
X_val_raw, y_val = X_train[idx_val], y_train[idx_val]

# scale with the mean and standard deviation of the training rows only
X_tr_scaled, mu_tr, sigma_tr = zscore_normalize_features(X_tr_raw)
X_val_scaled = (X_val_raw - mu_tr) / sigma_tr

# validation cost of each degree, filled in by the Exercise 4 cell (a new split starts empty)
val_cost = {}

print(f"Training set: {len(y_tr)} houses. Validation set: {len(y_val)} houses.")

The cell below plots the training set and the validation set.

In [ ]:
# plot the training set
fig,ax=plt.subplots(1,4,figsize=(12, 3),sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X_tr_raw[:,i],y_tr, label = 'target')
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("Price")
ax[0].legend()
fig.suptitle("Training set")
plt.show()


# plot the validation set
fig,ax=plt.subplots(1,4,figsize=(12, 3),sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X_val_raw[:,i],y_val, label = 'target')
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("Price")
ax[0].legend()
fig.suptitle("Validation set")
plt.show()

The function below runs gradient descent on the training set. At every iteration it also computes the cost on the validation set.

In [ ]:
def gradient_descent_val(X, y, X_val, y_val, w_in, b_in, cost_function, gradient_function, alpha, num_iters):
    """
    Performs batch gradient descent on the training set to learn w and b, and
    records the cost on the training set and on the validation set at every iteration

    Args:
      X (ndarray (m,n))   : training data, m examples with n features
      y (ndarray (m,))    : training target values
      X_val (ndarray (m_val,n)) : validation data
      y_val (ndarray (m_val,))  : validation target values
      w_in (ndarray (n,)) : initial model parameters
      b_in (scalar)       : initial model parameter
      cost_function       : function to compute cost
      gradient_function   : function to compute the gradient
      alpha (float)       : Learning rate
      num_iters (int)     : number of iterations to run gradient descent

    Returns:
      w (ndarray (n,))       : Updated values of parameters
      b (scalar)             : Updated value of parameter
      J_history_train (list) : training cost after each iteration
      J_history_val (list)   : validation cost after each iteration
      """

    # Lists to store the training and validation cost at each iteration, for the plots later
    J_history_train = []
    J_history_val = []
    w = copy.deepcopy(w_in)
    b = b_in

    for i in range(num_iters):

        # Calculate the gradient at the current w and b
        dj_db, dj_dw = gradient_function(X, y, w, b)

        # Update the parameters using alpha and the gradient
        w = w - alpha * dj_dw
        b = b - alpha * dj_db

        # Save cost J at each iteration
        if i<100000:      # prevent resource exhaustion
            J_history_train.append( cost_function(X, y, w, b))
            J_history_val.append( cost_function(X_val, y_val, w, b))

        # Print the cost 10 times during the run, or at every iteration if num_iters < 10
        if i% math.ceil(num_iters / 10) == 0:
            print(f"Iteration {i:4d}: Training cost {J_history_train[-1]:8.2f}, Validation cost {J_history_val[-1]:8.2f}   ")

    return w, b, J_history_train, J_history_val

### Exercise 4: Measuring Overfitting

- Set `degree` in the cell below and run it, then run the plot cell. Do this for degree = 1, 2, 3, 5, 10 and 20.
- Each run uses 10,000 iterations. The cell prints the training and the validation cost after 500, 1,000, 3,000 and 10,000 iterations, and the iteration where the validation cost was lowest.
- The lines that start with `Iteration` come from `gradient_descent_val`. As in Lecture 7, the line for iteration $i$ shows the cost after $i + 1$ steps. Use the lines that start with `After` for the tables.
- Fill in the two tables in the answer cell.
- The cell also stores the last validation cost of each degree in `val_cost`. The check of question 3 uses these values, so run it before you change the seed in question 4.

Then answer these questions.

1. How do the training cost and the validation cost change as the degree grows? What is happening at high degrees?
2. Look at degree 20 after 500, 1,000, 3,000 and 10,000 iterations, and at its plot. What happens to the validation cost when gradient descent runs longer? Propose a rule for when to stop.
3. Which degree would you choose? Write it in the cell after the answer cell and run the check.
4. Do this question at home. Go back to the split cell in Section 8.2 and change the seed to 0, and then to 4. Run the split cell and this exercise again for degree 1, 2 and 20. Does the degree you would choose change? Why?

In [ ]:
# Exercise 4: set the degree and run the cell
degree = 2
iterations = 10000
alpha = 0.01

# add size^2 ... size^degree, then scale with the training rows only
X_tr, mu_poly, sigma_poly = zscore_normalize_features(poly_features(X_tr_scaled, degree))
X_val = (poly_features(X_val_scaled, degree) - mu_poly) / sigma_poly

# run gradient descent on the training set
initial_w = np.zeros(X_tr.shape[1])
initial_b = 0.
w_final, b_final, J_hist_train, J_hist_val = gradient_descent_val(X_tr, y_tr, X_val, y_val, initial_w, initial_b,
                                                    compute_cost, compute_gradient,
                                                    alpha, iterations)
print(f"Degree {degree}")
for it in [500, 1000, 3000, 10000]:
    if it <= iterations:
        print(f"After {it:5d} iterations: training cost {J_hist_train[it-1]:8.2f}, validation cost {J_hist_val[it-1]:8.2f}")
# the position of the lowest validation cost. J_hist_val[best_it] is the cost after best_it + 1 steps
best_it = int(np.argmin(J_hist_val))
print(f"Lowest validation cost: {J_hist_val[best_it]:0.2f}, after {best_it + 1} iterations")
val_cost[degree] = J_hist_val[-1]   # used by the check at the end of Exercise 4

Expected output:

    Iteration    0: Training cost 67771.95, Validation cost 73088.07   
    Iteration 1000: Training cost   219.31, Validation cost   368.99   
    Iteration 2000: Training cost   196.37, Validation cost   298.96   
    Iteration 3000: Training cost   195.29, Validation cost   292.25   
    Iteration 4000: Training cost   195.24, Validation cost   291.18   
    Iteration 5000: Training cost   195.24, Validation cost   290.97   
    Iteration 6000: Training cost   195.24, Validation cost   290.93   
    Iteration 7000: Training cost   195.24, Validation cost   290.92   
    Iteration 8000: Training cost   195.24, Validation cost   290.92   
    Iteration 9000: Training cost   195.24, Validation cost   290.91   
    Degree 2
    After   500 iterations: training cost   310.46, validation cost   624.77
    After  1000 iterations: training cost   219.38, validation cost   369.19
    After  3000 iterations: training cost   195.29, validation cost   292.25
    After 10000 iterations: training cost   195.24, validation cost   290.91
    Lowest validation cost: 290.91, after 10000 iterations

This is the output for degree = 2, the value already in the cell.

In [ ]:
# plot the training and the validation cost versus iteration
start = 500
plt.plot(np.arange(start,iterations,1), J_hist_train[start:], label = "Training cost")
plt.plot(np.arange(start,iterations,1), J_hist_val[start:], label = "Validation cost")
plt.title(f"Degree {degree}: training and validation cost vs. iteration")
plt.ylabel('Cost')
plt.xlabel('iteration step')
plt.legend()
plt.show()

Write your results and answers here.

| Degree | Training cost after 10000 iterations | Validation cost after 10000 iterations |
|---|---|---|
| 1 | | |
| 2 | | |
| 3 | | |
| 5 | | |
| 10 | | |
| 20 | | |

The second table is for degree 20.

| Iterations | Training cost | Validation cost |
|---|---|---|
| 500 | | |
| 1000 | | |
| 3000 | | |
| 10000 | | |

1. ...
2. ...
3. ...
4. ...

In [ ]:
# Exercise 4: the degree you would choose
chosen_degree = None  # TODO: one of 1, 2, 3, 5, 10, 20

In [ ]:
# Check your result
assert chosen_degree is not None, "Write the degree you would choose in chosen_degree and run the cell."
degrees = [1, 2, 3, 5, 10, 20]
# the degrees that have no validation cost yet
missing = [d for d in degrees if d not in val_cost]
assert not missing, f"Run the Exercise 4 cell for every degree first. Not run yet with this split: {missing}"
assert chosen_degree in degrees, f"chosen_degree should be one of {degrees}."
lowest = np.nanmin([val_cost[d] for d in degrees])
assert val_cost[chosen_degree] <= 1.05 * lowest, "Look at the validation costs again. Choose a degree whose validation cost is the lowest or close to it."
print("Check passed")